In [0]:
from pyspark.sql.functions import lit,current_timestamp
from pyspark.sql.functions import trim, col, when
from datetime import datetime

pathVol=r"/Volumes/onedevcatalog/customer_data/customer-vol/rawfiles/"
fact_table = pathVol+r'fact_table.csv'

catalognm = 'onedevcatalog'
schema_b = 'bronze_one'
schema_s = 'silver_one'
schema_g = 'gold_one'
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
curr_timestamp = datetime.now()

### Fact Bronze Load

In [0]:
df = spark.read.csv(fact_table, header=True, inferSchema=True)
print(df.count())
df_upd = df.withColumn("load_time",lit(curr_timestamp))
df_upd = df_upd.withColumn("load_file",lit(fact_table))
#df_upd.display()
tablenm=f"{catalognm}.{schema_b}.fact_table"
df_upd.write.mode("overwrite").saveAsTable(tablenm)

### Fact Silver Load

In [0]:
tablenm_b=f"{catalognm}.{schema_b}.fact_table"
tablenm_s=f"{catalognm}.{schema_s}.fact_table"

df_b = spark.read.table(tablenm_b)
print("Bronze Count : ",df_b.count())

df_s = spark.read.table(tablenm_s)
print("Silver Count : ",df_s.count())

df_b_upd = df_b.drop("load_time","load_file")
df_b_upd = df_b_upd.withColumn("createdBy",lit(username)).withColumn("updatedBy",lit(username))
df_b_upd = df_b_upd.withColumn("createdTimestamp",lit(curr_timestamp)).withColumn("updatedTimestamp",lit(curr_timestamp))

string_cols = [c for c, t in df_b_upd.dtypes if t == "string"]

df_trimmed = df_b_upd.select(
    *[
        trim(col(c)).alias(c) if c in string_cols else col(c)
        for c in df_b_upd.columns
    ]
)

#df_trimmed.display()
df_trimmed.write.mode("overwrite").saveAsTable(tablenm_s)

### Fact Gold Load

In [0]:
tablenm_g=f"{catalognm}.{schema_g}.fact_table"
tablenm_s=f"{catalognm}.{schema_s}.fact_table"

df_g = spark.read.table(tablenm_g)
print("Gold Count : ",df_g.count())

df_s = spark.read.table(tablenm_s)
print("Silver Count : ",df_s.count())

df_s_upd = df_s.drop("createdBy","updatedBy","createdTimestamp","updatedTimestamp")
df_s_upd = df_s_upd.withColumn("createdBy",lit(username)).withColumn("updatedBy",lit(username))
df_s_upd = df_s_upd.withColumn("createdTimestamp",lit(curr_timestamp)).withColumn("updatedTimestamp",lit(curr_timestamp))

string_cols = [c for c, t in df_s_upd.dtypes if t == "string"]

df_cleaned = df_s_upd.select(
    *[
        when((col(c) == "") | (col(c).isin("null", "NULL")), None).otherwise(col(c)).alias(c)
        if c in string_cols else col(c)
        for c in df_s_upd.columns
    ]
)

#df_cleaned.display()
df_cleaned.write.mode("overwrite").saveAsTable(tablenm_g)